# AgroVision: Model Training

This notebook trains the crop disease classifier used by the AgroVision Streamlit app, using **MobileNetV2 transfer learning** on the complete **PlantVillage** dataset (38 classes, 14 crop species), loaded through **TensorFlow Datasets (TFDS)**.

**Before you start**
1. Select **Runtime > Change runtime type > T4 GPU**.
2. Select **Runtime > Run all**, then allow Google Drive access.

You do **not** need to upload a dataset. TFDS downloads the complete dataset by itself (about 870 MB) on the first run and keeps a copy in your Drive for later runs. Any ZIP you uploaded earlier is ignored.

**Honesty note:** this notebook has no stored results. Every number, chart, and metric appears only when you run it, and comes from your own model and test set.

| Output file | Used by |
|---|---|
| `agrovision_mobilenetv2.keras` | The app (`models/` folder) |
| `class_names.json` | The app (`models/` folder) |
| `class_mapping.json`, `metrics.json`, `classification_report.csv`, charts | Your report and faculty review |

## 1. Check the GPU and software versions

Training on a CPU would take far too long, so this cell stops with a clear message if no GPU is attached.

The versions are printed and saved because a Keras model file should be loaded by a Keras version that can read it. The app currently uses TensorFlow 2.21 and Keras 3.15.

In [ ]:
import sys
import keras
import tensorflow as tf

print("Python:    ", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Keras:     ", keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError(
        "No GPU found. Select Runtime > Change runtime type > T4 GPU, then run the notebook again."
    )
print("GPU:       ", gpus)
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Make TensorFlow Datasets available

Colab normally includes `tensorflow-datasets`. It is installed only if it is missing, because reinstalling packages unnecessarily can change versions that TensorFlow depends on.

In [ ]:
try:
    import tensorflow_datasets as tfds
except ImportError:
    !pip install -q tensorflow-datasets importlib_resources
    import tensorflow_datasets as tfds

print("TensorFlow Datasets:", tfds.__version__)

## 2. Import libraries and set the configuration

All the settings you might change are collected here.

- **`IMAGE_SIZE`** must stay 224 x 224, because that is what the app sends to the model.
- **The output file names** must stay the same, because `src/model.py` looks for those exact names.
- **`SEED`** makes the data split repeatable.

In [ ]:
import csv
import hashlib
import json
import random
import shutil
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import keras
import numpy as np
import tensorflow as tf
from PIL import Image

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

In [ ]:
# Google Drive locations
DRIVE_ROOT = Path("/content/drive/MyDrive/AgroVision")
OUTPUT_DIR = DRIVE_ROOT / "outputs"
TFDS_DRIVE_CACHE = DRIVE_ROOT / "tensorflow_datasets"  # Avoids downloading again later.

# Fast local disk used while training
TFDS_NAME = "plant_village"
TFDS_LOCAL_DIR = Path("/content/tensorflow_datasets")
DATASET_DIR = Path("/content/plantvillage_images")

# These must match src/preprocessing.py and src/model.py in the app.
IMAGE_SIZE = (224, 224)
MODEL_FILENAME = "agrovision_mobilenetv2.keras"
CLASS_NAMES_FILENAME = "class_names.json"

# Data
SEED = 42
SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
BATCH_SIZE = 32
EXPECTED_CLASS_COUNT = 38
EXPECTED_CROP_COUNT = 14
MIN_EXPECTED_IMAGES = 54_000  # Stops training on a partial dataset.

# Stage 1: train only the new classification head
HEAD_EPOCHS = 10
HEAD_LEARNING_RATE = 1e-3
DROPOUT_RATE = 0.2

# Stage 2: fine-tune the top of MobileNetV2
FINE_TUNE = True
FINE_TUNE_EPOCHS = 10
FINE_TUNE_LEARNING_RATE = 1e-5
FINE_TUNE_LAYERS = 30  # How many of the last MobileNetV2 layers to unfreeze.

USE_CLASS_WEIGHTS = True  # Gives rare classes more influence during training.

In [ ]:
keras.utils.set_random_seed(SEED)  # Seeds Python, NumPy, and TensorFlow together.

## 3. Connect Google Drive and load the complete dataset

**Why TensorFlow Datasets?** Some PlantVillage downloads are partial. For example, a 15-class ZIP would train a model that cannot recognize most of this project's classes. The TFDS `plant_village` builder provides the complete dataset, and TFDS verifies the download against a known checksum.

**What this section does:**

1. Mounts Google Drive, where results and checkpoints are saved.
2. Downloads and prepares the dataset with TFDS the first time, and keeps a copy in Drive. Later runs reuse that copy.
3. Stops with an error unless the dataset has exactly 38 classes across 14 crops and at least 54,000 images.
4. Exports the **original JPEG files** to Colab's fast local disk, using the class names the app expects.

**Why export the images?** TFDS can decode images itself, but this notebook needs to decode them in exactly the same way for every step, including the comparison with the app in Section 9. Keeping the original bytes avoids any re-encoding.

**Why rename six classes?** TFDS spells six class names differently from the app's treatment database:

| TFDS name | Name used by the app |
|---|---|
| `Cherry___healthy` | `Cherry_(including_sour)___healthy` |
| `Cherry___Powdery_mildew` | `Cherry_(including_sour)___Powdery_mildew` |
| `Corn___Cercospora_leaf_spot Gray_leaf_spot` | `Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot` |
| `Corn___Common_rust` | `Corn_(maize)___Common_rust_` |
| `Corn___healthy` | `Corn_(maize)___healthy` |
| `Corn___Northern_Leaf_Blight` | `Corn_(maize)___Northern_Leaf_Blight` |

Without this translation those classes would be predicted correctly, but the app could not find their treatment guidance.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs will be saved to:", OUTPUT_DIR)

In [ ]:
# Translates TFDS spellings to the class names used by the app.
TFDS_TO_APP_NAMES = {
    "Cherry___healthy": "Cherry_(including_sour)___healthy",
    "Cherry___Powdery_mildew": "Cherry_(including_sour)___Powdery_mildew",
    "Corn___Cercospora_leaf_spot Gray_leaf_spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn___Common_rust": "Corn_(maize)___Common_rust_",
    "Corn___healthy": "Corn_(maize)___healthy",
    "Corn___Northern_Leaf_Blight": "Corn_(maize)___Northern_Leaf_Blight",
}


def to_app_class_name(tfds_name):
    return TFDS_TO_APP_NAMES.get(tfds_name, tfds_name)


def check_dataset_is_complete(class_names, image_count):
    """Stop immediately if the dataset is a partial subset."""
    crop_count = len({name.split("___")[0] for name in class_names})
    if (len(class_names) != EXPECTED_CLASS_COUNT or crop_count != EXPECTED_CROP_COUNT
            or image_count < MIN_EXPECTED_IMAGES):
        raise ValueError(
            f"Incomplete dataset: {len(class_names)} classes, {crop_count} crops, "
            f"{image_count:,} images. Expected {EXPECTED_CLASS_COUNT} classes, "
            f"{EXPECTED_CROP_COUNT} crops, and at least {MIN_EXPECTED_IMAGES:,} images."
        )


def export_examples(examples, label_names, output_dir):
    """Write (encoded image, label index, file name) examples into class folders.

    The bytes are written unchanged, so the images are not re-compressed.
    Returns the number of images written.
    """
    count = 0
    for image_bytes, label, filename in examples:
        folder = output_dir / to_app_class_name(label_names[int(label)])
        folder.mkdir(parents=True, exist_ok=True)
        (folder / Path(filename).name).write_bytes(image_bytes)
        count += 1
    return count

In [ ]:
local_dataset = TFDS_LOCAL_DIR / TFDS_NAME
drive_dataset = TFDS_DRIVE_CACHE / TFDS_NAME

if drive_dataset.exists() and not local_dataset.exists():
    print("Copying the prepared dataset from Drive...")
    shutil.copytree(drive_dataset, local_dataset)

builder = tfds.builder(TFDS_NAME, data_dir=str(TFDS_LOCAL_DIR))
print("Downloading and preparing if needed. The first run can take a while...")
builder.download_and_prepare()

if not drive_dataset.exists():
    print("Saving a copy to Drive for future runs...")
    partial_copy = TFDS_DRIVE_CACHE / f"{TFDS_NAME}_incomplete"
    shutil.rmtree(partial_copy, ignore_errors=True)
    shutil.copytree(local_dataset, partial_copy)
    partial_copy.rename(drive_dataset)  # Only a finished copy gets the real name.

tfds_label_names = builder.info.features["label"].names
tfds_image_count = builder.info.splits["train"].num_examples

print("TFDS dataset:", builder.info.full_name)
print("Classes:", len(tfds_label_names))
print(f"Images:  {tfds_image_count:,}")
check_dataset_is_complete([to_app_class_name(n) for n in tfds_label_names], tfds_image_count)
print("Complete dataset confirmed.")

In [ ]:
# SkipDecoding returns each image's original JPEG bytes.
tfds_examples = builder.as_dataset(
    split="train", decoders={"image": tfds.decode.SkipDecoding()}
)
shutil.rmtree(DATASET_DIR, ignore_errors=True)

print("Exporting images to local disk...")
exported_count = export_examples(
    (
        (example["image"], example["label"], example["image/filename"].decode())
        for example in tfds.as_numpy(tfds_examples)
    ),
    tfds_label_names,
    DATASET_DIR,
)
print(f"Exported {exported_count:,} images to {DATASET_DIR}")
if exported_count != tfds_image_count:
    raise ValueError("The number of exported images does not match TFDS.")

## 4. Explore the classes

The exported folders are the classes, named `Crop___Condition`. Class names are **discovered from the folders and sorted**, and a class's position in this list becomes its model output index. That order is saved in `class_names.json`, so the app always uses the training order.

The list below is used only as a safety check. It confirms that the folder names match the app's treatment database, so that every prediction can find its guidance.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

class_names = sorted(path.name for path in DATASET_DIR.iterdir() if path.is_dir())
class_to_index = {name: index for index, name in enumerate(class_names)}

image_paths = {
    name: sorted(
        path for path in (DATASET_DIR / name).iterdir()
        if path.suffix.lower() in IMAGE_EXTENSIONS
    )
    for name in class_names
}
crops = sorted({name.split("___")[0] for name in class_names})

print(f"Classes: {len(class_names)}")
print(f"Crops:   {len(crops)}")
print(f"Images:  {sum(len(paths) for paths in image_paths.values()):,}")
print()
for index, name in enumerate(class_names):
    print(f"{index:>2}  {len(image_paths[name]):>5,}  {name}")

check_dataset_is_complete(class_names, sum(len(paths) for paths in image_paths.values()))
print("\nVerified: exactly 38 classes across 14 crops.")

In [ ]:
# Only used to check folder names. The model's class order comes from the folders above.
APP_CLASSES = {
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
}

missing = sorted(APP_CLASSES - set(class_names))
unexpected = sorted(set(class_names) - APP_CLASSES)
if len(class_names) != EXPECTED_CLASS_COUNT or missing or unexpected:
    raise ValueError(
        f"Expected the {EXPECTED_CLASS_COUNT} PlantVillage classes. "
        f"Missing: {missing}. Unexpected: {unexpected}."
    )
print("All 38 class folders match the app's treatment database.")

## 5. View sample images

Looking at the data before training helps you notice problems early. Notice that many PlantVillage photos have a plain, controlled background. Real field photos look different, which matters when interpreting the final results.

In [ ]:
sample_rng = random.Random(SEED)
sample_classes = sample_rng.sample(class_names, 12)

figure, axes = plt.subplots(3, 4, figsize=(14, 11))
for axis, name in zip(axes.flat, sample_classes):
    axis.imshow(Image.open(sample_rng.choice(image_paths[name])))
    axis.set_title(name.replace("___", "\n").replace("_", " "), fontsize=10)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 6. Count images per class

PlantVillage is **imbalanced**: some classes have far more images than others. A model could score well overall while doing poorly on rare classes. That is why we later use class weights and report per-class and macro-averaged metrics, not only accuracy.

In [ ]:
class_counts = {name: len(paths) for name, paths in image_paths.items()}
ordered = sorted(class_counts.items(), key=lambda item: item[1])

figure, axis = plt.subplots(figsize=(10, 11))
axis.barh([name.replace("___", ": ").replace("_", " ") for name, _ in ordered],
          [count for _, count in ordered], color="#1f6b45")
axis.set_xlabel("Number of images")
axis.set_title("Images per class")
axis.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=150)
plt.show()

print(f"Smallest class: {ordered[0][0]} ({ordered[0][1]:,})")
print(f"Largest class:  {ordered[-1][0]} ({ordered[-1][1]:,})")
print(f"Imbalance ratio: {ordered[-1][1] / ordered[0][1]:.1f} to 1")

## 7. Find damaged files and near-duplicate photos

**Why this matters:** if the same photo, or a nearly identical copy, appears in both training and test data, the model is tested on something it has already seen. This is called **data leakage**, and it makes results look better than they really are.

Each image gets a small visual fingerprint (a *difference hash*). Images with the same fingerprint form a group, and a whole group is later placed in a single split.

**Limitation:** this catches near-identical images only. Separate photos of the same leaf can still be divided between splits, so test results may remain somewhat optimistic.

In [ ]:
def image_fingerprint(path):
    """Return a 64-bit difference hash, or None if the image cannot be read."""
    try:
        with Image.open(path) as image:
            small = image.convert("L").resize((9, 8), Image.Resampling.BILINEAR)
    except OSError:
        return None
    pixels = np.asarray(small, dtype=np.int16)
    return np.packbits(pixels[:, 1:] > pixels[:, :-1]).tobytes().hex()


def assign_splits(records, ratios, seed):
    """Assign (path, label, group) records to splits, keeping each group together.

    Each class is divided close to the requested ratios (stratified), and groups
    are assigned one at a time to the split that is furthest below its target.
    """
    groups = defaultdict(list)
    for record in records:
        groups[record[2]].append(record)
    group_ids = sorted(groups)
    random.Random(seed).shuffle(group_ids)

    class_totals = Counter(label for _, label, _ in records)
    assigned_counts = {split: Counter() for split in ratios}
    assignments = {}
    for group_id in group_ids:
        members = groups[group_id]
        label = Counter(member[1] for member in members).most_common(1)[0][0]
        split = max(
            ratios,
            key=lambda name: ratios[name] * class_totals[label] - assigned_counts[name][label],
        )
        for path, member_label, _ in members:
            assignments[path] = split
            assigned_counts[split][member_label] += 1
    return assignments

In [ ]:
all_paths = [(path, name) for name, paths in image_paths.items() for path in paths]

print(f"Checking {len(all_paths):,} images. This can take a few minutes...")
with ThreadPoolExecutor(max_workers=8) as executor:
    fingerprints = list(executor.map(image_fingerprint, (path for path, _ in all_paths)))

damaged = [str(path) for (path, _), value in zip(all_paths, fingerprints) if value is None]
records = [
    (path, name, value)
    for (path, name), value in zip(all_paths, fingerprints)
    if value is not None
]

group_sizes = Counter(group for _, _, group in records)
labels_in_group = defaultdict(set)
for _, name, group in records:
    labels_in_group[group].add(name)

print(f"Damaged files skipped: {len(damaged)}")
print(f"Images in near-duplicate groups: {sum(size for size in group_sizes.values() if size > 1):,}")
print(f"Duplicate groups spanning different classes: "
      f"{sum(1 for names in labels_in_group.values() if len(names) > 1)}")

## 8. Create the training, validation, and test splits

| Split | Share | Purpose |
|---|---|---|
| Training | 70% | The model learns from these images |
| Validation | 15% | Chooses the best model and controls early stopping |
| Test | 15% | Used **once**, for the final results |

The split is **stratified** (each class keeps roughly these proportions) and **grouped** (near-duplicates stay together). The assertions below confirm that no image or fingerprint appears in two splits. The exact split is saved so the results can be reproduced.

In [ ]:
assignments = assign_splits(records, SPLIT_RATIOS, SEED)

splits = {name: {"paths": [], "labels": []} for name in SPLIT_RATIOS}
for path, name, _ in records:
    split = splits[assignments[path]]
    split["paths"].append(str(path))
    split["labels"].append(class_to_index[name])

# Prove that the splits do not overlap.
split_names = list(SPLIT_RATIOS)
fingerprint_of = {str(path): group for path, _, group in records}
for position, first in enumerate(split_names):
    for second in split_names[position + 1:]:
        assert not set(splits[first]["paths"]) & set(splits[second]["paths"])
        assert not ({fingerprint_of[p] for p in splits[first]["paths"]}
                    & {fingerprint_of[p] for p in splits[second]["paths"]})
assert sum(len(split["paths"]) for split in splits.values()) == len(records)

for name, split in splits.items():
    counts = Counter(split["labels"])
    print(f"{name:<6} {len(split['paths']):>7,} images   "
          f"smallest class {min(counts.values()):>4}   classes {len(counts)}")
    assert len(counts) == len(class_names), f"A class is missing from {name}."

with open(OUTPUT_DIR / "split_manifest.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["split", "class_name", "file"])
    for name, split in splits.items():
        for path, label in zip(split["paths"], split["labels"]):
            writer.writerow([name, class_names[label], Path(path).name])
print("Saved split_manifest.csv")

## 9. Build the input pipeline

The image pipeline must match the app, or the model will see different inputs in real use than in training.

| Step | Training notebook | Streamlit app (`src/preprocessing.py`) |
|---|---|---|
| Read JPEG | TensorFlow, `INTEGER_ACCURATE` | Pillow |
| Resize to 224 x 224 | Bilinear with antialiasing | Pillow `BILINEAR` |
| Scale to [-1, 1] | `mobilenet_v2.preprocess_input` | `pixels / 127.5 - 1` |

Scaling happens **once**, in this data pipeline and **not inside the model**. The app also scales images, so a scaling layer in the model would scale them twice and quietly hurt predictions.

The next cell compares the two pipelines on real images and stops if they differ meaningfully.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path):
    """Read and resize an image, returning float32 RGB values from 0 to 255."""
    raw = tf.io.read_file(path)
    image = tf.cond(
        tf.io.is_jpeg(raw),
        lambda: tf.io.decode_jpeg(raw, channels=3, dct_method="INTEGER_ACCURATE"),
        lambda: tf.io.decode_png(raw, channels=3),
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE,
                            method="bilinear", antialias=True)
    return tf.clip_by_value(image, 0.0, 255.0)


def app_preprocess(path):
    """The app's preprocessing, copied from src/preprocessing.py for comparison."""
    with Image.open(path) as image:
        resized = image.convert("RGB").resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32) / 127.5 - 1.0


# Training-only augmentation. Each epoch sees slightly different versions of each
# image, which reduces overfitting. Hue is not changed, because leaf color is
# evidence of disease. Validation and test images are never augmented.
geometric_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal_and_vertical"),
        keras.layers.RandomRotation(0.08, fill_mode="nearest"),
        keras.layers.RandomZoom(0.1, fill_mode="nearest"),
    ],
    name="augmentation",
)


def adjust_lighting(image, label):
    image = tf.image.random_brightness(image, max_delta=25.0)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    return tf.clip_by_value(image, 0.0, 255.0), label


def apply_geometry(images, labels):
    images = geometric_augmentation(images, training=True)
    return tf.clip_by_value(images, 0.0, 255.0), labels


def scale_for_model(images, labels):
    return keras.applications.mobilenet_v2.preprocess_input(images), labels


def make_dataset(paths, labels, training=False):
    """Build a batched dataset. Without training=True, the input order is kept."""
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(lambda path, label: (load_image(path), label),
                          num_parallel_calls=AUTOTUNE)
    if training:
        dataset = dataset.map(adjust_lighting, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    if training:
        dataset = dataset.map(apply_geometry, num_parallel_calls=AUTOTUNE)
    return dataset.map(scale_for_model, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
train_dataset = make_dataset(splits["train"]["paths"], splits["train"]["labels"], training=True)
val_dataset = make_dataset(splits["val"]["paths"], splits["val"]["labels"])
test_dataset = make_dataset(splits["test"]["paths"], splits["test"]["labels"])

images, labels = next(iter(val_dataset))
print("Batch shape:", images.shape, "| value range:",
      round(float(tf.reduce_min(images)), 3), "to", round(float(tf.reduce_max(images)), 3))

# Compare with the app's pipeline on real images. Differences are in 0-255 pixel levels.
differences = []
for path in random.Random(SEED).sample(splits["val"]["paths"], 100):
    training_input = keras.applications.mobilenet_v2.preprocess_input(load_image(path)).numpy()
    differences.append(np.abs(training_input - app_preprocess(path)) * 127.5)
mean_difference = float(np.mean(differences))
print(f"Average difference from the app: {mean_difference:.3f} pixel levels out of 255")
if mean_difference > 2.0:
    raise ValueError("Training and app preprocessing do not match. Do not train until this is fixed.")
print("Training preprocessing matches the app.")

In [ ]:
# Show one image with several random augmentations.
example_path = splits["train"]["paths"][0]
figure, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(load_image(example_path).numpy().astype("uint8"))
axes[0].set_title("Original")
for axis in axes[1:]:
    image, _ = adjust_lighting(load_image(example_path), 0)
    image, _ = apply_geometry(image[tf.newaxis], 0)
    axis.imshow(image[0].numpy().astype("uint8"))
    axis.set_title("Augmented")
for axis in axes:
    axis.axis("off")
plt.show()

## 10. Build the MobileNetV2 model

**Transfer learning** reuses a network already trained on ImageNet, whose early layers detect general shapes and textures. We remove its original 1000-class top and add our own head:

```text
Input (224 x 224 x 3, values in [-1, 1])
  -> MobileNetV2 base (ImageNet weights, frozen)
  -> Global average pooling
  -> Dropout
  -> Dense softmax with one output per class
```

- **Why MobileNetV2?** It is small and fast, which suits a web app.
- **Why freeze the base?** The new head starts random, and large early errors could otherwise damage the pretrained features.
- **`training=False`** keeps BatchNormalization layers in inference mode, including during fine-tuning, which is standard practice for transfer learning.

The app's Grad-CAM code expects exactly this simple chain with MobileNetV2 as one nested layer, so keep this structure.

In [ ]:
def build_model(class_count, weights="imagenet"):
    base = keras.applications.MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3), include_top=False, weights=weights
    )
    base.trainable = False

    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="leaf_image")
    x = base(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D(name="pooling")(x)
    x = keras.layers.Dropout(DROPOUT_RATE, name="dropout")(x)
    outputs = keras.layers.Dense(class_count, activation="softmax", name="predictions")(x)
    return keras.Model(inputs, outputs, name="agrovision_mobilenetv2")


def get_base(model):
    """Return the nested MobileNetV2 model."""
    return next(layer for layer in model.layers if isinstance(layer, keras.Model))


def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="sparse_categorical_crossentropy",  # Labels are class indices.
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )


def set_fine_tuning(model, layer_count):
    """Unfreeze the last layers of the base, keeping BatchNormalization frozen."""
    base = get_base(model)
    base.trainable = True
    for layer in base.layers[:-layer_count]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

In [ ]:
model = build_model(len(class_names))
compile_model(model, HEAD_LEARNING_RATE)
model.summary()

## 11. Stage 1: Train the classification head

Settings used:

- **Class weights** make mistakes on rare classes count for more, offsetting the imbalance from Section 6.
- **Checkpointing** saves the best model to Drive after any improving epoch, so a Colab disconnection does not lose everything.
- **Early stopping** ends training when validation loss stops improving, and restores the best weights.

Only validation data guides these decisions. The test set stays untouched.

In [ ]:
def make_class_weights(labels, class_count):
    """Weight each class by total / (classes x class size), so rare classes count more."""
    counts = np.bincount(labels, minlength=class_count)
    return {index: len(labels) / (class_count * counts[index]) for index in range(class_count)}


def training_callbacks(checkpoint_path):
    return [
        keras.callbacks.ModelCheckpoint(
            checkpoint_path, monitor="val_accuracy", mode="max", save_best_only=True
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
    ]

In [ ]:
class_weight = (
    make_class_weights(splits["train"]["labels"], len(class_names)) if USE_CLASS_WEIGHTS else None
)
HEAD_CHECKPOINT = OUTPUT_DIR / "checkpoint_head.keras"
FINE_TUNE_CHECKPOINT = OUTPUT_DIR / "checkpoint_fine_tuned.keras"

head_history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=HEAD_EPOCHS,
    class_weight=class_weight,
    callbacks=training_callbacks(HEAD_CHECKPOINT),
)
with open(OUTPUT_DIR / "history_head.json", "w") as file:
    json.dump({key: [float(v) for v in values] for key, values in head_history.history.items()}, file)

## 12. Stage 2: Fine-tune selected layers

Now the top layers of MobileNetV2 are unfrozen, so they can adapt to leaf disease patterns.

The learning rate is much lower (`1e-5`), because these layers already hold useful weights and large updates would damage them. Earlier layers, which detect general edges and textures, stay frozen, as do all BatchNormalization layers.

Set `FINE_TUNE = False` in Section 2 to skip this stage.

In [ ]:
if FINE_TUNE:
    # Start from the best head model, so this also works after a disconnection.
    model = keras.models.load_model(HEAD_CHECKPOINT, compile=False)
    set_fine_tuning(model, FINE_TUNE_LAYERS)
    compile_model(model, FINE_TUNE_LEARNING_RATE)
    print("Trainable weight tensors:", len(model.trainable_weights))

    fine_tune_history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=FINE_TUNE_EPOCHS,
        class_weight=class_weight,
        callbacks=training_callbacks(FINE_TUNE_CHECKPOINT),
    )
    with open(OUTPUT_DIR / "history_fine_tuned.json", "w") as file:
        json.dump({key: [float(v) for v in values]
                   for key, values in fine_tune_history.history.items()}, file)

## 13. Plot accuracy and loss

How to read these curves:

- Training and validation improving together is healthy.
- Training improving while validation gets worse suggests **overfitting**.
- The dashed line marks where fine-tuning began.

In [ ]:
stages = [json.loads((OUTPUT_DIR / "history_head.json").read_text())]
if FINE_TUNE:
    stages.append(json.loads((OUTPUT_DIR / "history_fine_tuned.json").read_text()))
history = {key: [value for stage in stages for value in stage[key]] for key in stages[0]}
epochs = range(1, len(history["accuracy"]) + 1)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for axis, metric, title in ((axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")):
    axis.plot(epochs, history[metric], label="Training", color="#1f6b45")
    axis.plot(epochs, history[f"val_{metric}"], label="Validation", color="#c26a1b")
    if FINE_TUNE:
        axis.axvline(len(stages[0]["accuracy"]) + 0.5, color="gray", linestyle="--",
                     label="Fine-tuning starts")
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

## 14. Evaluate on the test set

The better checkpoint is selected using **validation accuracy only**. It is then evaluated on the held-out test set, which played no part in training or model selection.

- **Accuracy:** the share of all test images classified correctly.
- **Precision:** of the images predicted as a class, how many really were that class?
- **Recall:** of the images truly in a class, how many did the model find?
- **F1-score:** the balance between precision and recall.
- **Macro average:** treats every class equally, so rare classes count fully.
- **Weighted average:** weights each class by its number of images.
- **Top-3 accuracy:** how often the correct class is among the app's three predictions.

To keep the test set honest, avoid repeatedly changing the model in response to these results. If you do tune further, use validation results.

In [ ]:
best_path = HEAD_CHECKPOINT
best_val_accuracy = max(stages[0]["val_accuracy"])
if FINE_TUNE and FINE_TUNE_CHECKPOINT.exists():
    fine_tuned_best = max(stages[1]["val_accuracy"])
    if fine_tuned_best > best_val_accuracy:
        best_path, best_val_accuracy = FINE_TUNE_CHECKPOINT, fine_tuned_best

print(f"Selected {best_path.name} (best validation accuracy {best_val_accuracy:.4f})")
model = keras.models.load_model(best_path, compile=False)

y_true = np.array(splits["test"]["labels"])
probabilities = model.predict(test_dataset, verbose=1)
assert len(probabilities) == len(y_true), "Predictions and labels are not aligned."
y_pred = probabilities.argmax(axis=1)
top3 = np.argsort(probabilities, axis=1)[:, -3:]

class_indices = list(range(len(class_names)))
macro = precision_recall_fscore_support(y_true, y_pred, average="macro",
                                        labels=class_indices, zero_division=0)
weighted = precision_recall_fscore_support(y_true, y_pred, average="weighted",
                                           labels=class_indices, zero_division=0)
metrics = {
    "test_images": int(len(y_true)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "top3_accuracy": float(np.mean([label in row for label, row in zip(y_true, top3)])),
    "macro_precision": float(macro[0]),
    "macro_recall": float(macro[1]),
    "macro_f1": float(macro[2]),
    "weighted_precision": float(weighted[0]),
    "weighted_recall": float(weighted[1]),
    "weighted_f1": float(weighted[2]),
    "selected_checkpoint": best_path.name,
    "best_val_accuracy": float(best_val_accuracy),
}
print(json.dumps(metrics, indent=2))

In [ ]:
report = classification_report(y_true, y_pred, labels=class_indices, target_names=class_names,
                               output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, labels=class_indices, target_names=class_names,
                            digits=3, zero_division=0))

with open(OUTPUT_DIR / "classification_report.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["class_name", "precision", "recall", "f1_score", "support"])
    for name in class_names:
        row = report[name]
        writer.writerow([name, row["precision"], row["recall"], row["f1-score"], int(row["support"])])

weakest = sorted(class_names, key=lambda name: report[name]["f1-score"])[:5]
print("Five lowest F1-scores:")
for name in weakest:
    print(f"  {report[name]['f1-score']:.3f}  {name}")

In [ ]:
matrix = confusion_matrix(y_true, y_pred, labels=class_indices)
normalized = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)  # Share of each true class.
short_names = [name.replace("___", ": ").replace("_", " ")[:38] for name in class_names]

figure, axis = plt.subplots(figsize=(17, 15))
image = axis.imshow(normalized, cmap="Greens", vmin=0, vmax=1)
axis.set_xticks(class_indices, short_names, rotation=90, fontsize=8)
axis.set_yticks(class_indices, short_names, fontsize=8)
axis.set_xlabel("Predicted class")
axis.set_ylabel("True class")
axis.set_title("Normalized confusion matrix (test set)")
figure.colorbar(image, shrink=0.7, label="Share of true class")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

off_diagonal = matrix.copy()
np.fill_diagonal(off_diagonal, 0)
print("Most common confusions:")
for position in np.argsort(off_diagonal, axis=None)[::-1][:5]:
    true_index, predicted_index = np.unravel_index(position, off_diagonal.shape)
    if off_diagonal[true_index, predicted_index]:
        print(f"  {off_diagonal[true_index, predicted_index]:>4}  "
              f"{class_names[true_index]} -> {class_names[predicted_index]}")

### Interpreting the results honestly

When you present these numbers:

- Quote the **test** metrics that your run printed above, not figures from other papers.
- PlantVillage images were mostly taken in controlled conditions, so accuracy on real field photos can be noticeably **lower** than the test result.
- Near-duplicate grouping reduces leakage, but cannot remove all of it, because separate photos of the same leaf may fall into different splits.
- The model only knows these 38 classes. It will still output one of them for an unrelated image.

## 15. Save the model and class mapping

| File | Contents |
|---|---|
| `agrovision_mobilenetv2.keras` | The selected model, without optimizer state, loaded by `src/model.py` |
| `class_names.json` | JSON list in which position equals the model's output index, read by the app |
| `class_mapping.json` | The same mapping written as `{"0": "Apple___Apple_scab", ...}`, for your report |
| `metrics.json` | The test metrics measured above |
| `training_metadata.json` | Versions and settings, for reproducibility |

`class_names` comes straight from the sorted folder list used to create the labels, so the saved order is exactly the training order.

In [ ]:
MODEL_PATH = OUTPUT_DIR / MODEL_FILENAME
CLASS_NAMES_PATH = OUTPUT_DIR / CLASS_NAMES_FILENAME

model.save(MODEL_PATH, include_optimizer=False)

with open(CLASS_NAMES_PATH, "w") as file:
    json.dump(class_names, file, indent=2)
with open(OUTPUT_DIR / "class_mapping.json", "w") as file:
    json.dump({str(index): name for index, name in enumerate(class_names)}, file, indent=2)
with open(OUTPUT_DIR / "metrics.json", "w") as file:
    json.dump(metrics, file, indent=2)
with open(OUTPUT_DIR / "training_metadata.json", "w") as file:
    json.dump({
        "tensorflow": tf.__version__,
        "keras": keras.__version__,
        "python": sys.version.split()[0],
        "image_size": list(IMAGE_SIZE),
        "seed": SEED,
        "split_ratios": SPLIT_RATIOS,
        "split_sizes": {name: len(split["paths"]) for name, split in splits.items()},
        "batch_size": BATCH_SIZE,
        "head_epochs_run": len(stages[0]["accuracy"]),
        "fine_tune_epochs_run": len(stages[1]["accuracy"]) if FINE_TUNE else 0,
        "fine_tune_layers": FINE_TUNE_LAYERS if FINE_TUNE else 0,
        "class_weights_used": USE_CLASS_WEIGHTS,
        "dataset_source": f"tfds:{builder.info.full_name}",
        "dataset_images": exported_count,
        "damaged_files_skipped": len(damaged),
    }, file, indent=2)

print(f"Model size: {MODEL_PATH.stat().st_size / 1e6:.1f} MB")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" ", path.name)

## 16. Verify the saved files work the way the app will use them

This cell repeats what `src/model.py` does: it reloads the saved files, checks their shapes, and predicts test images using **the app's preprocessing**.

It then compares those predictions with the evaluation pipeline. Strong agreement is evidence that the app will behave like the evaluated model.

In [ ]:
def check_saved_model(model, class_names):
    """The same shape checks that src/model.py applies when loading."""
    expected_input = (None, *IMAGE_SIZE, 3)
    if tuple(model.input_shape) != expected_input:
        raise ValueError(f"Input shape is {model.input_shape}, expected {expected_input}.")
    if model.output_shape[-1] != len(class_names):
        raise ValueError("The model's output count does not match class_names.json.")

In [ ]:
reloaded_model = keras.models.load_model(MODEL_PATH, compile=False)
reloaded_names = json.loads(CLASS_NAMES_PATH.read_text())
check_saved_model(reloaded_model, reloaded_names)
assert reloaded_names == class_names

sample_positions = random.Random(SEED).sample(range(len(y_true)), 300)
app_batch = np.stack([app_preprocess(splits["test"]["paths"][i]) for i in sample_positions])
app_predictions = reloaded_model.predict(app_batch, batch_size=BATCH_SIZE, verbose=0).argmax(axis=1)

agreement = float(np.mean(app_predictions == y_pred[sample_positions]))
print(f"App pipeline agrees with the evaluation pipeline on {agreement:.1%} of 300 test images.")
if agreement < 0.97:
    print("WARNING: agreement is low. Check preprocessing before trusting the app's predictions.")
else:
    print("The saved model and class names are ready for the app.")

## 17. Download the files for the app

Everything is already saved in `MyDrive/AgroVision/outputs/`. This cell also downloads the two required files to your computer. If your browser blocks the downloads, get them from Google Drive instead.

Then place them here in your project and restart the app:

```text
AgroVision/models/agrovision_mobilenetv2.keras
AgroVision/models/class_names.json
```

The sidebar changes from **Demo mode** to **Model connected**.

In [ ]:
from google.colab import files

files.download(str(MODEL_PATH))
files.download(str(CLASS_NAMES_PATH))